# AI IPS 결함 헌터

간단한 IPS 샘플 데이터를 여러 프롬프트 기반 Agent가 서로 다른 관점에서 검토하고, Mission Commander가 최종 조치 백로그를 생성합니다.

> 실습 데이터만 사용하며, 실제 군사·보안·개인정보 데이터는 입력하지 않습니다.

In [ ]:
# 최초 1회만 실행합니다.
# %pip install -U "crewai[openai]>=1.15,<2.0" python-dotenv

In [1]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv
from crewai import Agent, Crew, LLM, Process, Task

warnings.filterwarnings("ignore")

def find_env_file(start: Path) -> Path | None:
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    return None

env_path = find_env_file(Path.cwd())
if env_path is None:
    raise FileNotFoundError(".env 파일에 OPENAI_API_KEY를 설정하세요.")
load_dotenv(env_path, override=False)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다.")

model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
if not model_name.startswith("openai/"):
    model_name = f"openai/{model_name}"

llm = LLM(model=model_name, api_key=api_key, temperature=0.2)
print(f"사용 모델: {model_name}")

사용 모델: openai/gpt-4o-mini


## Agent 정의

각 Agent는 별도 도구나 학습 모델 없이 `role`, `goal`, `backstory` 프롬프트의 차이로 전문성을 갖습니다.

In [2]:
common_rules = (
    "입력으로 제공된 데이터만 근거로 사용한다. 없는 사실이나 수치를 만들지 않는다. "
    "정보가 부족하면 결함으로 단정하지 말고 '추가 확인 필요'라고 표시한다. "
    "모든 결과는 한국어로 작성하고 장문의 보고서 대신 우선순위 리스트로 작성한다."
)

fault_hunter = Agent(
    role="Fault Hunter - 반복고장 추적자",
    goal="정비이력에서 반복고장, 수리 후 재발, 장시간 정비가 필요한 결함 후보를 찾아 우선순위화한다.",
    backstory=(
        "같은 부품을 여러 번 교환하고도 고장이 재발한 사건을 겪은 집요한 정비분석가다. "
        "'수리 완료'라는 표현을 쉽게 믿지 않고 증상이 아닌 반복 패턴과 근본원인을 추적한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

supply_hunter = Agent(
    role="Supply Hunter - 결품 추적자",
    goal="재고, 최근 사용량, 조달기간을 비교해 정비 지연을 유발할 부품과 조달 액션을 찾는다.",
    backstory=(
        "핵심 부품 하나가 없어 여러 장비가 장기간 비가동된 상황을 경험한 보급 전문가다. "
        "창고의 전체 재고량보다 필요한 시점에 필요한 부품이 존재하는지를 중시한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

config_inspector = Agent(
    role="Configuration Detective - 형상·교범 탐정",
    goal="장비 형상, 장착 부품, BOM과 교범 버전 사이의 불일치 및 정보 누락을 찾는다.",
    backstory=(
        "정비사는 교범을 따랐지만 교범이 실제 장비 형상과 달라 문제가 발생한 사건을 조사했다. "
        "버전과 적용대상을 집요하게 대조하며 실제 불일치와 단순 정보 누락을 구분한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

mission_commander = Agent(
    role="Mission Commander - IPS 결함 헌터 지휘관",
    goal="전문 Agent의 후보를 검증·통합하여 임무 영향도와 시급성 기준의 실행 가능한 TOP 10 백로그를 만든다.",
    backstory=(
        "제한된 인력과 예산으로 전력 공백을 최소화해 온 IPS 총괄 책임자다. "
        "흥미로운 분석보다 지금 조치하지 않을 때 가장 큰 영향을 만드는 문제를 우선한다. "
        "중복 후보를 합치고, 근거가 약한 항목은 신뢰도를 낮추며, 최종 결정은 사람에게 남긴다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

## 간단한 샘플 데이터

실제 데이터베이스 없이 텍스트·CSV에서 복사한 소규모 목록으로 실습할 수 있습니다.

In [3]:
ips_data = r'''
[기준일] 2026-07-23

[정비이력]
WO-101 | 장비 A-01 | 형상 B | 2026-06-03 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 8시간 | 완료
WO-102 | 장비 A-03 | 형상 B | 2026-06-18 | 고장 E-104 | 커넥터 재체결 | 수리 3시간 | 완료
WO-103 | 장비 A-01 | 형상 B | 2026-07-02 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 9시간 | 완료
WO-104 | 장비 A-07 | 형상 C | 2026-07-10 | 고장 H-210 | 유압필터 HF-10 교환 | 수리 5시간 | 완료
WO-105 | 장비 A-01 | 형상 B | 2026-07-19 | 고장 E-104 | 원인 미상, 재기동 | 수리 2시간 | 관찰 중

[부품재고]
CM-20 | 제어모듈 | 현재고 1 | 최근 60일 사용 2 | 조달기간 90일 | 대체품 없음
HF-10 | 유압필터 | 현재고 42 | 최근 60일 사용 1 | 조달기간 14일 | 대체품 HF-11
CN-05 | 커넥터 | 현재고 3 | 최근 60일 사용 1 | 조달기간 미입력 | 대체품 없음

[형상/BOM/교범]
장비 A-01 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.2
장비 A-03 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.1
BOM 형상 B | 제어모듈 CM-20 | 적용교범 TM-v1.2
장비 A-07 | 형상 C | 장착부품 HF-10 | 적용교범 TM-v2.0
BOM 형상 C | 유압필터 HF-11 | 적용교범 TM-v2.0

[교범 발췌]
TM-v1.1: 이상 발생 시 커넥터를 적절히 체결하고 필요시 제어모듈을 교환한다.
TM-v1.2: E-104 발생 시 CN-05 체결상태 확인 후 CM-20 진단시험을 수행한다. 완료기준은 미기재.
TM-v2.0: 형상 C의 유압필터 품번은 HF-11이다.
'''

print(ips_data)


[기준일] 2026-07-23

[정비이력]
WO-101 | 장비 A-01 | 형상 B | 2026-06-03 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 8시간 | 완료
WO-102 | 장비 A-03 | 형상 B | 2026-06-18 | 고장 E-104 | 커넥터 재체결 | 수리 3시간 | 완료
WO-103 | 장비 A-01 | 형상 B | 2026-07-02 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 9시간 | 완료
WO-104 | 장비 A-07 | 형상 C | 2026-07-10 | 고장 H-210 | 유압필터 HF-10 교환 | 수리 5시간 | 완료
WO-105 | 장비 A-01 | 형상 B | 2026-07-19 | 고장 E-104 | 원인 미상, 재기동 | 수리 2시간 | 관찰 중

[부품재고]
CM-20 | 제어모듈 | 현재고 1 | 최근 60일 사용 2 | 조달기간 90일 | 대체품 없음
HF-10 | 유압필터 | 현재고 42 | 최근 60일 사용 1 | 조달기간 14일 | 대체품 HF-11
CN-05 | 커넥터 | 현재고 3 | 최근 60일 사용 1 | 조달기간 미입력 | 대체품 없음

[형상/BOM/교범]
장비 A-01 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.2
장비 A-03 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.1
BOM 형상 B | 제어모듈 CM-20 | 적용교범 TM-v1.2
장비 A-07 | 형상 C | 장착부품 HF-10 | 적용교범 TM-v2.0
BOM 형상 C | 유압필터 HF-11 | 적용교범 TM-v2.0

[교범 발췌]
TM-v1.1: 이상 발생 시 커넥터를 적절히 체결하고 필요시 제어모듈을 교환한다.
TM-v1.2: E-104 발생 시 CN-05 체결상태 확인 후 CM-20 진단시험을 수행한다. 완료기준은 미기재.
TM-v2.0: 형상 C의 유압필터 품번은 HF-11이다.



## Task 및 출력 형식 설정

In [4]:
fault_task = Task(
    description=(
        "다음 IPS 데이터 중 [정비이력]을 중심으로 분석하세요.\n{ips_data}\n\n"
        "동일·유사 고장 반복, 수리 후 재발, 원인 미상, 장시간 수리 후보를 최대 5건 찾으세요. "
        "각 항목을 [심각도] 제목 | 근거 | 영향 | 권고조치 | 신뢰도 형식의 목록으로 출력하세요."
    ),
    expected_output="근거가 명시된 반복고장 후보 목록 최대 5건",
    agent=fault_hunter,
)

supply_task = Task(
    description=(
        "다음 IPS 데이터 중 [부품재고]와 정비이력을 함께 분석하세요.\n{ips_data}\n\n"
        "결품, 장기조달, 반복고장 연계, 과다재고 후보를 최대 5건 찾으세요. "
        "각 항목을 [심각도] 제목 | 근거 | 영향 | 권고조치 | 신뢰도 형식의 목록으로 출력하세요."
    ),
    expected_output="근거가 명시된 보급 위험 및 비용 기회 목록 최대 5건",
    agent=supply_hunter,
)

config_task = Task(
    description=(
        "다음 IPS 데이터의 [형상/BOM/교범]과 [교범 발췌]를 교차 검토하세요.\n{ips_data}\n\n"
        "형상·품번·교범 버전 불일치, 모호한 절차, 완료기준 누락을 최대 5건 찾으세요. "
        "각 항목을 [심각도] 제목 | 비교근거 | 영향 | 권고조치 | 신뢰도 형식의 목록으로 출력하세요."
    ),
    expected_output="근거가 명시된 형상 및 교범 결함 후보 목록 최대 5건",
    agent=config_inspector,
)

command_task = Task(
    description=(
        "세 전문 Agent의 결과를 통합해 최종 IPS 결함·조치 백로그를 만드세요. "
        "같은 원인의 후보는 합치고, 근거 없는 항목은 제외하며, 심각도는 CRITICAL/HIGH/MEDIUM/LOW만 사용하세요. "
        "우선순위는 안전·임무 영향, 영향 장비 수, 반복성, 시급성, 조치 효과 순으로 판단하세요.\n\n"
        "반드시 다음 열을 가진 마크다운 표만 출력하세요.\n"
        "| 순위 | 심각도 | 결함/위험 | 데이터 근거 | 권고 조치 | 담당 | 기한 | 신뢰도 |\n"
        "담당은 정비/보급/형상관리/기술교범 중 선택하고, 기한은 즉시/7일/30일/추가 확인 중 하나로 정하세요. "
        "최대 10건만 출력하고 표 다음에는 설명문을 추가하지 마세요."
    ),
    expected_output="8개 열을 갖춘 한국어 마크다운 IPS 결함·조치 우선순위 표 최대 10건",
    agent=mission_commander,
    context=[fault_task, supply_task, config_task],
)

In [5]:
crew = Crew(
    agents=[fault_hunter, supply_hunter, config_inspector, mission_commander],
    tasks=[fault_task, supply_task, config_task, command_task],
    process=Process.sequential,
    verbose=True,
)

## 실행 및 최종 백로그 출력

아래 셀은 API를 호출합니다. 실행 결과는 설명문이 아닌 우선순위 표로 표시됩니다.

In [6]:
from IPython.display import Markdown, display

result = await crew.kickoff_async(inputs={"ips_data": ips_data})
final_backlog = result.raw
display(Markdown("## IPS 결함·조치 우선순위\n\n" + final_backlog))

# 필요하면 전문 Agent별 중간 결과를 확인할 수 있습니다.
# print(fault_task.output.raw)
# print(supply_task.output.raw)
# print(config_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9aed7ffc-09ce-4c39-8681-d9d8ebcb88f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 IPS 데이터 중 [정비이력]을 중심으로 분석하세요.                                                     │
│                                                                                                                 │
│  [기준일] 2026-07-23                                                                                            │
│                                                                                                                 │
│  [정비이력]                                                                                                     │
│  WO-101 | 장비 A-01 | 형상 B | 2026-06-03 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 8시간 | 완료                │
│  WO-102 | 장비 A-03 | 형상 B | 2026-06-18 | 고장 E-104 | 커넥터 재체결 | 수리 3시간 | 완료                      │
│  WO-103 | 장비 A-01 | 형상 B | 2026-07-02 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 9시간 | 완료                │
│  WO-104 | 장비 A-07 | 형상 C | 2026-07-10 | 고장 H-210 | 유압필터 HF-10 교환 | 수리 5시간 | 완료                │
│  WO-105 | 장비 A-01 | 형상 B | 2026-07-19 | 고장 E-104 | 원인 미상, 재기동 | 수리 2시간 | 관찰 중               │
│                                                                                                                 │
│  [부품재고]                                                                                                     │
│  CM-20 | 제어모듈 | 현재고 1 | 최근 60일 사용 2 | 조달기간 90일 | 대체품 없음                                   │
│  HF-10 | 유압필터 | 현재고 42 | 최근 60일 사용 1 | 조달기간 14일 | 대체품 HF-11                                 │
│  CN-05 | 커넥터 | 현재고 3 | 최근 60일 사용 1 | 조달기간 미입력 | 대체품 없음                                   │
│                                                                                                                 │
│  [형상/BOM/교범]                                                                                                │
│  장비 A-01 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.2                                                         │
│  장비 A-03 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.1                                                         │
│  BOM 형상 B | 제어모듈 CM-20 | 적용교범 TM-v1.2                                                                 │
│  장비 A-07 | 형상 C | 장착부품 HF-10 | 적용교범 TM-v2.0                                                         │
│  BOM 형상 C | 유압필터 HF-11 | 적용교범 TM-v2.0                                                                 │
│                                                                                                                 │
│  [교범 발췌]                                                                                                    │
│  TM-v1.1: 이상 발생 시 커넥터를 적절히 체결하고 필요시 제어모듈을 교환한다.                                     │
│  TM-v1.2: E-104 발생 시 CN-05 체결상태 확인 후 CM-20 진단시험을 수행한다. 완료기준은 미기재.                    │
│  TM-v2.0: 형상 C의 유압필터 품번은 HF-11이다.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  동일·유사 고장 반복, 수리 후 재발, 원인 미상, 장시간 수리 후보를 최대 5건 찾으세요. 각 항목을 [심각도] 제목 |  │
│  근거 | 영향 | 권고조치 | 신뢰도 형식의 목록으로 출력하세요.                                                    │
│  ID: c4c8ee32-4ef1-4f2b-8de2-97e3edc029f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fault Hunter - 반복고장 추적자                                                                          │
│                                                                                                                 │
│  Task: 다음 IPS 데이터 중 [정비이력]을 중심으로 분석하세요.                                                     │
│                                                                                                                 │
│  [기준일] 2026-07-23                                                                                            │
│                                                                                                                 │
│  [정비이력]                                                                                                     │
│  WO-101 | 장비 A-01 | 형상 B | 2026-06-03 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 8시간 | 완료                │
│  WO-102 | 장비 A-03 | 형상 B | 2026-06-18 | 고장 E-104 | 커넥터 재체결 | 수리 3시간 | 완료                      │
│  WO-103 | 장비 A-01 | 형상 B | 2026-07-02 | 고장 E-104 | 제어모듈 CM-20 교환 | 수리 9시간 | 완료                │
│  WO-104 | 장비 A-07 | 형상 C | 2026-07-10 | 고장 H-210 | 유압필터 HF-10 교환 | 수리 5시간 | 완료                │
│  WO-105 | 장비 A-01 | 형상 B | 2026-07-19 | 고장 E-104 | 원인 미상, 재기동 | 수리 2시간 | 관찰 중               │
│                                                                                                                 │
│  [부품재고]                                                                                                     │
│  CM-20 | 제어모듈 | 현재고 1 | 최근 60일 사용 2 | 조달기간 90일 | 대체품 없음                                   │
│  HF-10 | 유압필터 | 현재고 42 | 최근 60일 사용 1 | 조달기간 14일 | 대체품 HF-11                                 │
│  CN-05 | 커넥터 | 현재고 3 | 최근 60일 사용 1 | 조달기간 미입력 | 대체품 없음                                   │
│                                                                                                                 │
│  [형상/BOM/교범]                                                                                                │
│  장비 A-01 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.2                                                         │
│  장비 A-03 | 형상 B | 장착부품 CM-20 | 적용교범 TM-v1.1                                                         │
│  BOM 형상 B | 제어모듈 CM-20 | 적용교범 TM-v1.2                                                                 │
│  장비 A-07 | 형상 C | 장착부품 HF-10 | 적용교범 TM-v2.0                                                         │
│  BOM 형상 C | 유압필터 HF-11 | 적용교범 TM-v2.0                                                                 │
│                                                                                                                 │
│  [교범 발췌]                                                                                                    │
│  TM-v1.1: 이상 발생 시 커넥터를 적절히 체결하고 필요시 제어모듈을 교환한다.                                     │
│  TM-v1.2: E-104 발생 시 CN-05 체결상태 확인 후 CM-20 진단시험을 수행한다. 완료기준은 미기재.                    │
│  TM-v2.0: 형상 C의 유압필터 품번은 HF-11이다.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  동일·유사 고장 반복, 수리 후 재발, 원인 미상, 장시간 수리 후보를 최대 5건 찾으세요. 각 항목을 [심각도] 제목 |  │
│  근거 | 영향 | 권고조치 | 신뢰도 형식의 목록으로 출력하세요.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fault Hunter - 반복고장 추적자                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. [심각도] 반복 고장 E-104 | 근거 | 장비 A-01에서 2026-06-03, 2026-07-02에 제어모듈 CM-20 교환 후에도         │
│  2026-07-19에 원인 미상으로 재발 | 영향 | 장비 A-01의 가동 중단 및 추가 수리 필요 | 권고조치 | 제어모듈         │
│  CM-20의 진단시험 및 교환 후 재발 원인 분석 필요 | 신뢰도 | 높음                                                │
│                                                                                                                 │
│  2. [심각도] 반복 고장 E-104 | 근거 | 장비 A-03에서 2026-06-18에 커넥터 재체결 후에도 E-104 고장 발생 | 영향 |  │
│  장비 A-03의 가동 중단 및 추가 수리 필요 | 권고조치 | 커넥터 CN-05의 체결상태 점검 및 교환 필요 | 신뢰도 |      │
│  중간                                                                                                           │
│                                                                                                                 │
│  3. [심각도] 원인 미상 고장 | 근거 | 장비 A-01에서 2026-07-19에 원인 미상으로 재기동 후 관찰 중 | 영향 | 장비   │
│  A-01의 불안정한 작동 및 추가 고장 가능성 | 권고조치 | 원인 분석을 위한 추가 진단 및 모니터링 필요 | 신뢰도 |   │
│  중간                                                                                                           │
│                                                                                                                 │
│  4. [심각도] 장시간 수리 | 근거 | 장비 A-01의 고장 E-104 수리 시 8시간, 9시간 소요 | 영향 | 수리 비용 증가 및   │
│  가동 중단 시간 증가 | 권고조치 | 수리 프로세스 개선 및 부품 교환 기준 재검토 필요 | 신뢰도 | 높음              │
│                                                                                                                 │
│  5. [심각도] 부품 재고 부족 | 근거 | 제어모듈 CM-20의 현재고 1개로 최근 60일 사용 2회 | 영향 | 추가 고장 시     │
│  즉각적인 수리 불가 | 권고조치 | 제어모듈 CM-20의 재고 확보 및 조달 기간 단축 필요 | 신뢰도 | 높음              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Mission Commander - IPS 결함 헌터 지휘관                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | 순위 | 심각도 | 결함/위험       | 데이터 근거                                                                │
│  | 권고 조치                                          | 담당   | 기한   | 신뢰도 |                              │
│  |------|--------|------------------|-------------------------------------------------------------------------  │
│  --|---------------------------------------------------|--------|--------|--------|                             │
│  | 1    | CRITICAL | 반복 고장 E-104  | 장비 A-01에서 2026-06-03, 2026-07-02에 제어모듈 CM-20 교환 후에도       │
│  2026-07-19에 원인 미상으로 재발 | 제어모듈 CM-20의 진단시험 및 교환 후 재발 원인 분석 필요 | 정비   | 즉시     │
│  | 높음   |                                                                                                     │
│  | 2    | HIGH   | 반복 고장 E-104  | 장비 A-03에서 2026-06-18에 커넥터 재체결 후에도 E-104 고장 발생           │
│  | 커넥터 CN-05의 체결상태 점검 및 교환 필요          | 정비   | 7일    | 중간   |                              │
│  | 3    | HIGH   | 장시간 수리      | 장비 A-01의 고장 E-104 수리 시 8시간, 9시간 소요                          │
│  | 수리 프로세스 개선 및 부품 교환 기준 재검토 필요   | 보급   | 30일   | 높음   |                              │
│  | 4    | HIGH   | 부품 재고 부족   | 제어모듈 CM-20의 현재고 1개로 최근 60일 사용 2회                       |  │
│  제어모듈 CM-20의 재고 확보 및 조달 기간 단축 필요  | 보급   | 즉시   | 높음   |                                │
│  | 5    | MEDIUM | 원인 미상 고장   | 장비 A-01에서 2026-07-19에 원인 미상으로 재기동 후 관찰 중                │
│  | 원인 분석을 위한 추가 진단 및 모니터링 필요        | 기술교범 | 추가 확인 | 중간   |                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 세 전문 Agent의 결과를 통합해 최종 IPS 결함·조치 백로그를 만드세요. 같은 원인의 후보는 합치고, 근거      │
│  없는 항목은 제외하며, 심각도는 CRITICAL/HIGH/MEDIUM/LOW만 사용하세요. 우선순위는 안전·임무 영향, 영향 장비     │
│  수, 반복성, 시급성, 조치 효과 순으로 판단하세요.                                                               │
│                                                                                                                 │
│  반드시 다음 열을 가진 마크다운 표만 출력하세요.                                                                │
│  | 순위 | 심각도 | 결함/위험 | 데이터 근거 | 권고 조치 | 담당 | 기한 | 신뢰도 |                                 │
│  담당은 정비/보급/형상관리/기술교범 중 선택하고, 기한은 즉시/7일/30일/추가 확인 중 하나로 정하세요. 최대        │
│  10건만 출력하고 표 다음에는 설명문을 추가하지 마세요.                                                          │
│  Agent: Mission Commander - IPS 결함 헌터 지휘관                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9aed7ffc-09ce-4c39-8681-d9d8ebcb88f7                                                                       │
│  Final Output: | 순위 | 심각도 | 결함/위험       | 데이터 근거                                                  │
│  | 권고 조치                                          | 담당   | 기한   | 신뢰도 |                              │
│  |------|--------|------------------|-------------------------------------------------------------------------  │
│  --|---------------------------------------------------|--------|--------|--------|                             │
│  | 1    | CRITICAL | 반복 고장 E-104  | 장비 A-01에서 2026-06-03, 2026-07-02에 제어모듈 CM-20 교환 후에도       │
│  2026-07-19에 원인 미상으로 재발 | 제어모듈 CM-20의 진단시험 및 교환 후 재발 원인 분석 필요 | 정비   | 즉시     │
│  | 높음   |                                                                                                     │
│  | 2    | HIGH   | 반복 고장 E-104  | 장비 A-03에서 2026-06-18에 커넥터 재체결 후에도 E-104 고장 발생           │
│  | 커넥터 CN-05의 체결상태 점검 및 교환 필요          | 정비   | 7일    | 중간   |                              │
│  | 3    | HIGH   | 장시간 수리      | 장비 A-01의 고장 E-104 수리 시 8시간, 9시간 소요                          │
│  | 수리 프로세스 개선 및 부품 교환 기준 재검토 필요   | 보급   | 30일   | 높음   |                              │
│  | 4    | HIGH   | 부품 재고 부족   | 제어모듈 CM-20의 현재고 1개로 최근 60일 사용 2회                       |  │
│  제어모듈 CM-20의 재고 확보 및 조달 기간 단축 필요  | 보급   | 즉시   | 높음   |                                │
│  | 5    | MEDIUM | 원인 미상 고장   | 장비 A-01에서 2026-07-19에 원인 미상으로 재기동 후 관찰 중                │
│  | 원인 분석을 위한 추가 진단 및 모니터링 필요        | 기술교범 | 추가 확인 | 중간   |                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## IPS 결함·조치 우선순위

| 순위 | 심각도 | 결함/위험       | 데이터 근거                                                                 | 권고 조치                                          | 담당   | 기한   | 신뢰도 |
|------|--------|------------------|---------------------------------------------------------------------------|---------------------------------------------------|--------|--------|--------|
| 1    | CRITICAL | 반복 고장 E-104  | 장비 A-01에서 2026-06-03, 2026-07-02에 제어모듈 CM-20 교환 후에도 2026-07-19에 원인 미상으로 재발 | 제어모듈 CM-20의 진단시험 및 교환 후 재발 원인 분석 필요 | 정비   | 즉시   | 높음   |
| 2    | HIGH   | 반복 고장 E-104  | 장비 A-03에서 2026-06-18에 커넥터 재체결 후에도 E-104 고장 발생          | 커넥터 CN-05의 체결상태 점검 및 교환 필요          | 정비   | 7일    | 중간   |
| 3    | HIGH   | 장시간 수리      | 장비 A-01의 고장 E-104 수리 시 8시간, 9시간 소요                        | 수리 프로세스 개선 및 부품 교환 기준 재검토 필요   | 보급   | 30일   | 높음   |
| 4    | HIGH   | 부품 재고 부족   | 제어모듈 CM-20의 현재고 1개로 최근 60일 사용 2회                       | 제어모듈 CM-20의 재고 확보 및 조달 기간 단축 필요  | 보급   | 즉시   | 높음   |
| 5    | MEDIUM | 원인 미상 고장   | 장비 A-01에서 2026-07-19에 원인 미상으로 재기동 후 관찰 중              | 원인 분석을 위한 추가 진단 및 모니터링 필요        | 기술교범 | 추가 확인 | 중간   |

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯